# Dataset inventory
Prepare the code package and inspect an **already anonymized** image directory. This notebook does not anonymize images. Set the paths below; the database remains in your own Drive.

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess
PROJECT_ROOT = Path('/content/DMAE_Colab_Repository')
if not PROJECT_ROOT.exists():
    local = Path.cwd()
    if (local / 'src' / 'amd_oct').exists():
        PROJECT_ROOT = local
    else:
        from google.colab import files
        print('Upload DMAE_Colab_Repository.zip (code package, not the image database).')
        uploaded = files.upload()
        package = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if package is None:
            raise ValueError('Select the code ZIP package.')
        destination = Path('/content').resolve()
        with zipfile.ZipFile(package) as archive:
            for member in archive.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination):
                    raise ValueError('Unsafe archive path.')
            archive.extractall(destination)
if not (PROJECT_ROOT / 'src' / 'amd_oct').is_dir():
    raise FileNotFoundError('Set PROJECT_ROOT to the extracted code-package directory.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Code root:', PROJECT_ROOT)


In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/AMD_OCT') if USE_GOOGLE_DRIVE else Path('/content/AMD_OCT')
DATASET_DIR = BASE_DIR / 'data'  # Contains Normal, iORA, cORA, iRORA, cRORA folders.
CACHE_DIR = BASE_DIR / 'features'
MODEL_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
for directory in (CACHE_DIR, MODEL_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
from amd_oct.data import inventory_images, EXPECTED_COUNTS
inventory = inventory_images(DATASET_DIR, RESULTS_DIR / 'dataset_inventory.csv')
counts = inventory.groupby('category').size().rename('observed')
import pandas as pd
comparison = pd.DataFrame({'manuscript_count': EXPECTED_COUNTS}).join(counts)
comparison['difference'] = comparison['observed'].fillna(0) - comparison['manuscript_count']
display(comparison)
display(inventory.groupby(['width', 'height']).size().rename('images'))
print('Images:', len(inventory))
print('Patient count cannot be recovered from sequential image filenames alone.')
